# Stimuli Sequence Generation

This notebook generates fully counterbalanced stimuli sequences for participants.

**Agent Types:**
- Agent 1: low danger, high difficulty
- Agent 2: high danger, low difficulty
- Agent 3: low danger, low difficulty
- Agent 4: high danger, high difficulty

**Design:**
- 2 danger types (high and low) x 2 map difficulty (high and low) → 4 agents
- 4 agents → 4! = 24 possible agent orders
- 6 videos per cell (each danger x map has 6 variants): high and low amplitude trajectories coming from 3 models
- Each participant sees two trajectories from the SAME model
- 4 color variants (red, blue, yellow, green) - counterbalanced using Latin square (cycling)
- 3 question types (danger, map_difficulty, enjoyment) - each participant answers ONE question type

- Map numbers and colors are assigned to agents based on their position in the agent order
- Each participant watches 4 videos (4 agents, 2 trajectories from the same agent, 4 colors)
- Total sequences: 24 (agent orders) × 6 (A1 vs A2, A2 vs A1, B1 vs B2, B2 vs B1, C1 vs C2) × 3 (questions) = 432 unique participants
- We'll run it twice x 2 = 864 unique participants

→ 192 participants × 4 videos per participant = 768 total video trials


In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations_with_replacement, permutations
import random

# Set seed for reproducibility
random.seed(42)
np.random.seed(42)


## Define Video Database


In [ ]:
import os
import re
from pathlib import Path

# Base URL for videos
BASE_URL = "https://engaging-movies.s3.us-east-2.amazonaws.com/cogsci-stimuli/"

# Parse video filenames from stimuli/videos folder

videos_dir = Path('videos')
video_files = [f for f in os.listdir(videos_dir) if f.endswith('.mp4')]

# Pattern to extract information from filename
pattern = r'(\w+)_trajectory_(low|high)_amplitude_(low|high)_danger_[\d.]+_(low|high)_difficulty_[\d.]+_(\w+)\.mp4'

# Define agent mapping: agent number -> (danger_level, difficulty_level)
AGENT_TYPES = {
    1: ('low', 'high'),    # low danger, high difficulty
    2: ('high', 'low'),    # high danger, low difficulty
    3: ('low', 'low'),     # low danger, low difficulty
    4: ('high', 'high')    # high danger, high difficulty
}

# Group videos by (danger_level, difficulty_level) then map to agent numbers
agents_by_type = {}

for filename in video_files:
    match = re.match(pattern, filename)
    if match:
        model_name = str(match.group(1))
        amplitude = match.group(2)  # 'low' or 'high'
        danger_level = match.group(3)  # 'low' or 'high'
        difficulty_level = match.group(4)  # 'low' or 'high'
        color = match.group(5)  # 'red', 'blue', 'yellow', or 'green'
        
        # Create agent type key (danger_level, difficulty_level)
        agent_type = (danger_level, difficulty_level)
        
        # Initialize agent type if not exists
        if agent_type not in agents_by_type:
            agents_by_type[agent_type] = {
                'name': f'{danger_level}_danger_{difficulty_level}_difficulty',
                'danger_level': danger_level,
                'difficulty_level': difficulty_level,
                'videos': {}
            }
        
        # Add video with key (model_name, amplitude, color)
        video_key = (model_name, amplitude, color)
        agents_by_type[agent_type]['videos'][video_key] = filename

# Now create the final agents dictionary with integer keys
agents = {}
for agent_num, agent_type in AGENT_TYPES.items():
    if agent_type in agents_by_type:
        agents[agent_num] = agents_by_type[agent_type]

# Extract unique values for reference
model_names = sorted(set(key[0] for agent in agents.values() for key in agent['videos'].keys()))
amplitudes = sorted(set(key[1] for agent in agents.values() for key in agent['videos'].keys()))
colors = sorted(set(key[2] for agent in agents.values() for key in agent['videos'].keys()))

print(f"Total agents: {len(agents)}")
print(f"\nAgent mapping:")
for agent_num in sorted(agents.keys()):
    agent = agents[agent_num]
    print(f"  Agent {agent_num}: {agent['danger_level']} danger, {agent['difficulty_level']} difficulty")

print(f"\nModel names: {model_names}")
print(f"Amplitudes: {amplitudes}")
print(f"Color variants: {colors}")
print(f"Videos per agent: {len(next(iter(agents.values()))['videos']) if agents else 0}")
print(f"\nSample agent structure:")
for agent_num in sorted(agents.keys())[:1]:
    agent_data = agents[agent_num]
    print(f"  Agent {agent_num}: {agent_data['name']}")
    print(f"  First 3 videos: {list(agent_data['videos'].items())[:3]}")


## Generate All Counterbalancing Conditions

We'll enumerate all possible combinations of agent orders, damage levels, and color orders (Latin square).


In [ ]:
# Generate all permutations of agent orders (4! = 24)
# Using agent numbers 1, 2, 3, 4 instead of tuples
agent_orders = list(permutations([1, 2, 3, 4]))

print(f"Number of agent order permutations: {len(agent_orders)}")
print(f"First 3 agent orders: {agent_orders[:3]}")

# Define Latin square for color assignment and map_set assignment
def latin_square(lst):
    """Generate a Latin square from a list"""
    n = len(lst)
    return [tuple(lst[i:] + lst[:i]) for i in range(n)]


# Generate Latin square for colors
COLORS = ['red', 'blue', 'yellow', 'green']
color_orders = latin_square(COLORS)

print(f"\nColor orders (Latin square - 4 rotations):")
for i, color_order in enumerate(color_orders, 1):
    print(f"  {i}. {color_order}")

# Define question types (3 total: danger, map_difficulty, enjoyment)
QUESTIONS = ['danger', 'map_difficulty', 'enjoyment']
print(f"\nQuestion types (3 total): {QUESTIONS}")

# Define trajectory pair combinations
# Each participant sees 2 trajectories from the SAME model
# Models: A (mr_2_1_10), B (mr_5_2_20), C (unlimited)
# Amplitudes: 1 = low, 2 = high
# 6 combinations: A1 vs A2, A2 vs A1, B1 vs B2, B2 vs B1, C1 vs C2, C2 vs C1

MODEL_MAPPING = {
    'A': 'mr_2_1_10',
    'B': 'mr_5_2_20', 
    'C': 'unlimited'
}

# Create trajectory pair combinations
# Format: (model_letter, amplitude1, amplitude2) where amplitude1 and amplitude2 are 'low' or 'high'
trajectory_pairs = [
    ('A', 'low', 'high'),   # A1 vs A2
    ('A', 'high', 'low'),   # A2 vs A1
    ('B', 'low', 'high'),   # B1 vs B2
    ('B', 'high', 'low'),   # B2 vs B1
    ('C', 'low', 'high'),   # C1 vs C2
    ('C', 'high', 'low'),   # C2 vs C1
]

print(f"\nTrajectory pair combinations (6 total):")
for i, (model_letter, amp1, amp2) in enumerate(trajectory_pairs, 1):
    model_name = MODEL_MAPPING[model_letter]
    print(f"  {i}. {model_letter}: {model_name} ({amp1} vs {amp2})")


## Generate Participant Sequences

For each participant:
1. Assign a specific agent order (24 permutations)
2. Assign ONE trajectory pair (6 combinations: A1 vs A2, A2 vs A1, B1 vs B2, B2 vs B1, C1 vs C2, C2 vs C1)
   - Each participant sees 2 trajectories from the SAME model
   - Amplitudes: first two agents use amplitude1, last two agents use amplitude2
   - A1 pattern: (low, low, high, high), A2 pattern: (high, high, low, low)
3. Assign ONE question type (3 question types: danger, map_difficulty, enjoyment)
4. Assign a color order (from 4 Latin square rotations, cycling)
5. Colors are mapped to agents based on their position in the agent order
6. Each participant watches 4 videos (one per agent, with amplitude pattern from the trajectory pair)


In [ ]:
def generate_participant_sequence(participant_id, agent_order, trajectory_pair, color_order, question):
    """
    Generate a video sequence for one participant.
    
    Args:
        participant_id: Unique identifier (e.g., 1, 2, 3, ...)
        agent_order: Tuple of 4 agent numbers (e.g., (1, 2, 3, 4) or (3, 1, 4, 2))
        trajectory_pair: Tuple of (model_letter, amplitude1, amplitude2) e.g., ('A', 'low', 'high')
        color_order: Tuple of 4 colors in order (e.g., ('red', 'blue', 'yellow', 'green'))
        question: Question type for this participant (e.g., 'danger', 'map_difficulty', 'enjoyment')
    
    Returns:
        DataFrame with columns: participantID, videoID, agentNumber, dangerLevel, difficultyLevel, 
                                modelName, amplitude, mapNumber, color, question, videoUrl
    """
    participant_videos = []
    
    model_letter, amplitude1, amplitude2 = trajectory_pair
    model_name = MODEL_MAPPING[model_letter]
    
    # Generate 4 videos (one per agent)
    # Map numbers and colors are assigned based on position in agent_order
    # Amplitudes: first two agents (video_id 1, 2) use amplitude1, last two (video_id 3, 4) use amplitude2
    # This gives patterns: A1 = (low, low, high, high), A2 = (high, high, low, low)
    for video_id, (agent_num, color) in enumerate(zip(agent_order, color_order), start=1):
        agent_info = agents[agent_num]
        danger_level = agent_info['danger_level']
        difficulty_level = agent_info['difficulty_level']
        
        # First two videos use amplitude1, last two use amplitude2
        amplitude = amplitude1 if video_id <= 2 else amplitude2
        
        # Get the video filename using (model_name, amplitude, color) key
        video_key = (model_name, amplitude, color)
        
        # Check if this key exists, if not try the other amplitude
        if video_key not in agent_info['videos']:
            # Try with the other amplitude
            other_amplitude = amplitude2 if amplitude == amplitude1 else amplitude1
            video_key = (model_name, other_amplitude, color)
        
        if video_key not in agent_info['videos']:
            # If still not found, try map 1 or 2
            for alt_map in [1, 2]:
                alt_key = (model_name, amplitude, alt_map, color)
                if alt_key in agent_info['videos']:
                    video_key = alt_key
                    break
        
        video_filename = agent_info['videos'][video_key]
        video_url = BASE_URL + video_filename
        
        participant_videos.append({
            'participantID': participant_id,
            'videoID': video_id,
            'agentNumber': agent_num,
            'dangerLevel': danger_level,
            'difficultyLevel': difficulty_level,
            'modelName': model_name,
            'amplitude': amplitude,
            'color': color,
            'question': question,
            'videoUrl': video_url
        })
    
    return pd.DataFrame(participant_videos)

# Test with one example participant
test_agent_order = (1, 2, 3, 4)
test_trajectory_pair = ('A', 'low', 'high')
test_participant = generate_participant_sequence(
    participant_id=1, 
    agent_order=test_agent_order,
    trajectory_pair=test_trajectory_pair,
    color_order=('red', 'blue', 'yellow', 'green'),
    question='danger'
)
print("Example participant 1:")
print(f"  Agent order: {test_agent_order}")
print(f"  Agent details:")
for agent_num in test_agent_order:
    agent = agents[agent_num]
    print(f"    Agent {agent_num}: {agent['danger_level']} danger, {agent['difficulty_level']} difficulty")
print(f"  Trajectory pair: {test_trajectory_pair[0]}1 vs {test_trajectory_pair[0]}2 ({test_trajectory_pair[1]} vs {test_trajectory_pair[2]})")
print(f"  Model: {MODEL_MAPPING[test_trajectory_pair[0]]}")
print(f"\nSequence (4 videos total):")
print(test_participant)


## Generate All Participant Sequences


In [ ]:
all_sequences = []
participant_counter = 1

###############################
# For testing, only use a subset of agent orders
number_participants_pilot = None  # Set to None to generate the full dataset
###############################

# Store conditions for each participant
participant_conditions = {}

# Helper for determining participant limit
def participant_limit_reached(counter, limit):
    return (limit is not None) and (counter > limit)

# Iterate through all combinations of agent orders, trajectory pairs, questions, map orders, and color orders
# Total: 24 (agent orders) × 6 (trajectory pairs) × 3 (questions) = 432 participants
for _ in range (2):
    for agent_order in agent_orders:
        for trajectory_pair in trajectory_pairs:
            for question in QUESTIONS:
                if participant_limit_reached(participant_counter, number_participants_pilot):
                    break

                participant_id = participant_counter  # 1, 2, ..., N

                # Assign and color_order using modulo rotation
                color_order = color_orders[(participant_counter-1) % len(color_orders)]

                # Store the conditions for this participant
                model_letter, amp1, amp2 = trajectory_pair
                participant_conditions[participant_id] = {
                    'agentOrderCondition': list(agent_order),
                    'trajectoryPair': {
                        'model': MODEL_MAPPING[model_letter],
                        'amplitude1': amp1,
                        'amplitude2': amp2
                    },
                    'colorOrderCondition': list(color_order),
                    'questionCondition': question
                }

                # Generate sequence
                sequence = generate_participant_sequence(
                    participant_id,
                    agent_order,
                    trajectory_pair,
                    color_order,
                    question
                )

                all_sequences.append(sequence)
                participant_counter += 1
            if participant_limit_reached(participant_counter, number_participants_pilot):
                break
        if participant_limit_reached(participant_counter, number_participants_pilot):
            break

# Combine all sequences into one DataFrame
final_df = pd.concat(all_sequences, ignore_index=True)

print(f"✓ Generated sequences for {participant_counter - 1} participants")
print(f"Expected: 24 (agent orders) × 6 (trajectory pairs) × 3 (questions) = 432 participants")
print(f"Total rows in dataset: {len(final_df)}")
print(f"Videos per participant: {final_df.groupby('participantID').size().iloc[0]}")

print(f"\nFirst 5 rows of final dataset:")
print(final_df.head())

## Save to CSV


In [ ]:
# Save to CSV file
output_filename = 'stimuli_sequences_cogsci.csv'
final_df.to_csv(output_filename, index=False)

print(f"✓ Successfully saved to: {output_filename}")
print(f"\nFile details:")
print(f"  - Columns: {list(final_df.columns)}")
print(f"  - Shape: {final_df.shape} (rows × columns)")
print(f"  - Participants: 1 to {final_df['participantID'].nunique()}")
print(f"  - Videos per participant: {final_df.groupby('participantID').size().iloc[0]}")


## Transform to MongoDB Format

Convert from long format (one row per video) to wide format (one document per participant with nested trials array).


In [ ]:
# Generate MongoDB documents with conditions
def transform_to_mongodb_format_with_conditions(df, conditions_dict):
    """
    Transform long format DataFrame to MongoDB-ready format with counterbalancing conditions.
    
    Args:
        df: DataFrame with columns [participantID, videoID, agentNumber, dangerLevel, difficultyLevel, 
                                    modelName, amplitude, color, question, videoUrl]
        conditions_dict: Dictionary mapping participantID to agentOrderCondition, trajectoryPair, 
                        colorOrderCondition, and questionCondition
    
    Returns:
        List of dictionaries ready for MongoDB insertion
    """
    mongo_docs = []
    
    for participant_id in sorted(df['participantID'].unique()):
        # Get all trials for this participant
        participant_data = df[df['participantID'] == participant_id].sort_values('videoID')
        
        # Convert trials to list of dictionaries
        trials = []
        for _, row in participant_data.iterrows():
            trial = {
                'videoID': int(row['videoID']),
                'agentNumber': int(row['agentNumber']),
                'dangerLevel': row['dangerLevel'],
                'difficultyLevel': row['difficultyLevel'],
                'modelName': row['modelName'],
                'amplitude': row['amplitude'],
                'color': row['color'],
                'question': row['question'],
                'videoUrl': row['videoUrl']
            }
            trials.append(trial)
        
        # Create participant document with conditions
        doc = {
            'participantID': int(participant_id),
            'numTrials': len(trials),
            'agentOrderCondition': conditions_dict[participant_id]['agentOrderCondition'],
            'trajectoryPair': conditions_dict[participant_id]['trajectoryPair'],
            'colorOrderCondition': conditions_dict[participant_id]['colorOrderCondition'],
            'questionCondition': conditions_dict[participant_id]['questionCondition'],
            'trials': trials
        }
        
        mongo_docs.append(doc)
    
    return mongo_docs

# Transform the data WITH conditions
mongo_documents = transform_to_mongodb_format_with_conditions(final_df, participant_conditions)

print(f"✓ Generated {len(mongo_documents)} participant documents WITH conditions")
print(f"\nExample document structure (Participant 1):")
print(f"  participantID: {mongo_documents[0]['participantID']}")
print(f"  numTrials: {mongo_documents[0]['numTrials']}")
print(f"  agentOrderCondition: {mongo_documents[0]['agentOrderCondition']}")
print(f"  trajectoryPair: {mongo_documents[0]['trajectoryPair']}")
print(f"  colorOrderCondition: {mongo_documents[0]['colorOrderCondition']}")
print(f"  questionCondition: {mongo_documents[0]['questionCondition']}")
print(f"  trials: [{len(mongo_documents[0]['trials'])} trial objects]")
print(f"\nFirst trial example:")
print(f"  {mongo_documents[0]['trials'][0]}")


## Save MongoDB Documents to JSON


In [ ]:
import json

# Save as JSON for MongoDB import
json_filename = 'stimuli_sequences_mongodb.json'
with open(json_filename, 'w') as f:
    json.dump(mongo_documents, f, indent=2)

print(f"✓ Saved MongoDB documents to: {json_filename}")
print(f"\nFile contains {len(mongo_documents)} documents")
print(f"Each document has:")
print(f"  - participantID (integer)")
print(f"  - numTrials (integer) = {mongo_documents[0]['numTrials']}")
print(f"  - agentOrderCondition (array of 4 agent numbers: 1-4)")
print(f"  - trajectoryNumber (integer)")
print(f"  - colorOrderCondition (array of 4 colors)")
print(f"  - questionCondition (string) - one of: danger, map_difficulty, enjoyment, competence")
print(f"  - trials (array of {mongo_documents[0]['numTrials']} trial objects)")
print(f"\nEach trial object contains:")
print(f"  - videoID, agentNumber, dangerLevel, difficultyLevel, trajectoryNumber, color, question, videoUrl")
